# 05 — Baseline Classifiers

**Goal:** Establish baseline classification performance for Q1 (species classification)
and Q2 (ARG burden prediction) using Logistic Regression, under both standard and
phylogenetically-grouped cross-validation.

**Why "baseline first"?**
Any more complex model (Random Forest in Phase 8, XGBoost in Phase 9) must beat this
by a margin that justifies its added complexity. A Random Forest with the same accuracy
as Logistic Regression has not earned its existence.

**What this notebook delivers — Q1 reporting format (pre-registered in `docs/pre_analysis_plan.md §4`):**

| | Full features (274 dp_*) | Specificity-filtered (<0.70 std) |
|---|---|---|
| Standard StratifiedKFold | preliminary reference | preliminary reference |
| Phylogenetic GroupedStratifiedKFold | new — Phase 7 | **★ PRIMARY RESULT** |

The [filtered features + grouped CV] cell is the single primary Q1 result reported in the manuscript.
All others provide context.

**Q2:** LR for ARG burden prediction, per species, under GroupedStratifiedKFold.


## Section 2 — Imports and configuration

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import (
    balanced_accuracy_score, f1_score,
    confusion_matrix, roc_auc_score,
)
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore")

ROOT = Path("..")
PROC = ROOT / "data" / "processed"
FIG  = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
(ROOT / "results").mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS     = 5
N_BOOT       = 2000

print("Imports OK.")


Imports OK.


## Section 3 — Load data

Two files from Phase 6:
- `feature_matrix.parquet` — 878 × 631 (features + labels), indexed by genome accession
- `cv_groups.parquet` — 878 × 1 (`phylogroup` column), the output of Phase 6

We align them on index (accession) and verify no mismatches.


In [2]:
fm        = pd.read_parquet(PROC / "feature_matrix_3460.parquet")
cv_groups = pd.read_parquet(PROC / "cv_groups_3460.parquet")

# Align cv_groups to fm's index order (both have the same 878 accessions, possibly different order)
assert set(fm.index) == set(cv_groups.index), "Accession sets do not match!"
cv_groups = cv_groups.reindex(fm.index)     # reorder to match feature matrix

groups  = cv_groups["phylogroup"].values    # 878-element array — group labels for CV
species = fm["species"].values              # Q1 target labels

print(f"Feature matrix:  {fm.shape}")
print(f"Phylogroups:     {cv_groups['phylogroup'].nunique()} unique groups")
print(f"\nSpecies counts:")
print(pd.Series(species).value_counts().to_string())
print(f"\nQ2 label counts:")
print(fm["arg_burden_tertile"].value_counts().to_string())

Feature matrix:  (878, 632)
Phylogroups:     95 unique groups

Species counts:
saureus        150
paeruginosa    150
abaumannii     150
efaecium       150
ecloaceae      146
kpneumoniae    132

Q2 label counts:
arg_burden_tertile
low_ARG     325
high_ARG    289
mid_ARG     264


## Section 4 — Define feature sets

**Full set (274 features):** All `dp_*` binary columns.

**Specificity-filtered set:** Remove "taxonomic marker" features whose per-species
prevalence standard deviation (normalised by 0.5) exceeds 0.70. These features are
near-universal in one species and near-absent in others — they inflate Q1 accuracy
by encoding species identity rather than defence architecture.

**How the specificity score is computed:**
For each feature, compute the fraction of genomes in each of the 6 species that carry it.
Take the standard deviation of those 6 fractions. Divide by 0.5 (the maximum possible
standard deviation for a feature present in 100% of one species and 0% of the rest).
Score near 1 = strong species marker. Score near 0 = evenly distributed.


In [3]:
dp_cols = sorted([c for c in fm.columns if c.startswith("dp_")])
print(f"Total dp_ features: {len(dp_cols)}")

# Per-species prevalence: fraction of each species' genomes carrying each feature
sp_prev = fm.groupby("species")[dp_cols].mean()          # shape: 6 × 274

# Specificity score: std of per-species prevalence, normalised to [0,1]
spec_score = sp_prev.std() / 0.5                         # Series indexed by feature

# Taxonomic markers
marker_features = spec_score[spec_score >= 0.70].index.tolist()
print(f"Taxonomic marker features (score ≥ 0.70): {len(marker_features)}")
print()
print(f"{'Feature':<40} {'Score':>6}  Dominant species (prevalence)")
print("-" * 75)
for f in sorted(marker_features, key=lambda x: -spec_score[x]):
    dominant = sp_prev[f].idxmax()
    print(f"  {f:<38} {spec_score[f]:>6.3f}  {dominant} ({sp_prev[f].max():.0%})")

filtered_cols = [c for c in dp_cols if c not in marker_features]
print(f"\nFull set:     {len(dp_cols)} features")
print(f"Filtered set: {len(filtered_cols)} features  ({len(marker_features)} removed)")

X_full = fm[dp_cols].values.astype(np.float32)
X_filt = fm[filtered_cols].values.astype(np.float32)


Total dp_ features: 274
Taxonomic marker features (score ≥ 0.70): 9

Feature                                   Score  Dominant species (prevalence)
---------------------------------------------------------------------------
  dp_padloc_PDC-S07                       1.035  kpneumoniae (100%)
  dp_padloc_PDC-S12                       1.018  kpneumoniae (99%)
  dp_PD-T4-6                              0.995  paeruginosa (99%)
  dp_VSPR                                 0.995  ecloaceae (99%)
  dp_AbiE                                 0.865  kpneumoniae (92%)
  dp_padloc_PDC-S04                       0.853  kpneumoniae (99%)
  dp_df_gcu233                            0.811  saureus (99%)
  dp_padloc_SoFic                         0.771  paeruginosa (87%)
  dp_RM_Type_IV                           0.705  kpneumoniae (91%)

Full set:     274 features
Filtered set: 265 features  (9 removed)


## Section 5 — CV helper functions and classifier setup

**`run_cv`**: runs k-fold CV, collecting all individual predictions (not just fold averages).
Returns (y_true, y_pred, fold_scores). Using individual predictions rather than fold averages
is important for getting accurate CIs — fold-level CIs from only 5 numbers are too noisy.

**`bootstrap_ci`**: resamples the 878 individual (y_true, y_pred) pairs 2000 times to
estimate the 95% CI for balanced accuracy. This is the standard method for CI on CV results.

**DummyClassifier (`strategy="stratified"`):** The null baseline. It ignores all features
and randomly assigns labels according to the class proportions seen in training. For 6
roughly balanced species (≈150 each), this gives balanced accuracy ≈ 1/6 = 0.167.
Any real model must substantially beat this.


In [4]:
def run_cv(X, y, cv, groups=None, clf=None):
    """Run k-fold CV; return concatenated (y_true, y_pred, fold_scores)."""
    all_true, all_pred, fold_scores = [], [], []
    for train_idx, test_idx in cv.split(X, y, groups=groups):
        m = clone(clf)                        # fresh copy — no state leaks between folds
        m.fit(X[train_idx], y[train_idx])
        y_pred = m.predict(X[test_idx])
        all_true.extend(y[test_idx])
        all_pred.extend(y_pred)
        fold_scores.append(balanced_accuracy_score(y[test_idx], y_pred))
    return np.array(all_true), np.array(all_pred), np.array(fold_scores)


def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=N_BOOT, rs=RANDOM_STATE):
    """Bootstrap 95% CI by resampling (y_true, y_pred) pairs."""
    rng = np.random.RandomState(rs)
    n = len(y_true)
    boot = [metric_fn(y_true[idx := rng.choice(n, n, replace=True)],
                      y_pred[idx])
            for _ in range(n_boot)]
    return np.percentile(boot, [2.5, 97.5])


# Classifier — multi_class removed (deprecated in sklearn 1.7; handled natively in 1.8)
lr = LogisticRegression(
    C=1.0,
    penalty="l2",
    solver="lbfgs",        # efficient L-BFGS optimiser — good for multinomial
    max_iter=2000,
    random_state=RANDOM_STATE,
)

# Null baseline
null_clf = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)

# CV strategies
skf  = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print("Setup complete.")
print(f"LR: C={lr.C}, penalty={lr.penalty}, solver={lr.solver}")
print(f"StratifiedKFold:       random assignment, stratified by species")
print(f"StratifiedGroupKFold:  phylogroups assigned whole to folds, stratified by species")

Setup complete.
LR: C=1.0, penalty=l2, solver=lbfgs
StratifiedKFold:       random assignment, stratified by species
StratifiedGroupKFold:  phylogroups assigned whole to folds, stratified by species


## Section 6 — Q1: Standard StratifiedKFold (reference — NOT primary)

Standard CV randomly shuffles genomes into 5 folds, ensuring each fold has roughly
equal species proportions. **This does not protect against clone contamination** — two
near-identical genomes from the same hospital outbreak can end up in different folds.

We run it here to:
1. Reproduce the preliminary numbers (should match 0.984 / 0.936 from earlier notebook).
2. Fill the top row of the 2×2 table within this notebook for direct comparison.

These numbers are **reference only**. They will not be cited as primary results.


In [5]:
print("=== Q1: Standard StratifiedKFold (reference) ===\n")

results_q1 = {}   # (feat_name, cv_name) → (bac, ci_lo, ci_hi, fold_scores)

for feat_name, X in [("full_{}"    .format(len(dp_cols)), X_full), ("filtered_{}".format(len(filtered_cols)), X_filt)]:
    yt, yp, fscores = run_cv(X, species, skf, groups=None, clf=lr)
    bac = balanced_accuracy_score(yt, yp)
    ci  = bootstrap_ci(yt, yp, balanced_accuracy_score)
    f1  = f1_score(yt, yp, average="macro")
    results_q1[(feat_name, "standard_CV")] = (bac, ci[0], ci[1], fscores)

    print(f"Feature set: {feat_name}")
    print(f"  Fold scores: {[round(s, 3) for s in fscores]}")
    print(f"  Mean balanced acc: {bac:.3f}  95% CI: [{ci[0]:.3f}, {ci[1]:.3f}]")
    print(f"  Macro-F1:          {f1:.3f}")
    print()


=== Q1: Standard StratifiedKFold (reference) ===



Feature set: full_274
  Fold scores: [np.float64(0.983), np.float64(0.994), np.float64(1.0), np.float64(0.971), np.float64(0.989)]
  Mean balanced acc: 0.988  95% CI: [0.980, 0.994]
  Macro-F1:          0.987



Feature set: filtered_266
  Fold scores: [np.float64(0.943), np.float64(0.971), np.float64(0.948), np.float64(0.959), np.float64(0.93)]
  Mean balanced acc: 0.950  95% CI: [0.935, 0.964]
  Macro-F1:          0.951



## Section 7 — Q1: StratifiedGroupKFold (phylogenetic correction)

**How StratifiedGroupKFold assigns folds:**

It treats this as an optimisation problem: assign each of the 95 phylogroups to one of 5 folds
such that:
1. No phylogroup is split across folds (hard constraint).
2. Class proportions (species fractions) are as balanced as possible across folds (soft goal).

It proceeds greedily: for each phylogroup (processed largest first), assign it to the fold
that currently has the largest *deficit* in the species this phylogroup mostly belongs to.
This cannot guarantee perfect balance — because phylogroups come in very different sizes —
so fold sizes will be unequal. The fold structure print below shows exactly what gets assigned
where.

**What to watch for:**
- Fold sizes: expect unequal sizes due to large phylogroups
- Per-species representation per fold: should have all 6 species in every fold
- Delta vs standard CV: the drop quantifies how much the preliminary result was inflated
  by clone contamination


In [6]:
print("=== Q1: StratifiedGroupKFold (phylogenetic correction) ===\n")
print("--- Fold structure ---")

sp_abbrevs = {sp: sp[:2].upper() for sp in sorted(set(species))}
sp_order   = sorted(set(species))
fold_sizes = []

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X_full, species, groups=groups)):
    fold_sizes.append(len(test_idx))
    sp_counts = pd.Series(species[test_idx]).value_counts()
    pg_count  = pd.Series(groups[test_idx]).nunique()
    sp_str = "  ".join(f"{sp_abbrevs[sp]}:{sp_counts.get(sp,0)}" for sp in sp_order)
    print(f"  Fold {fold+1}: {len(test_idx):>4} genomes | {pg_count:>3} phylogroups | {sp_str}")

print(f"\nFold sizes: {fold_sizes}")
print(f"  Range: {min(fold_sizes)}–{max(fold_sizes)} genomes  "
      f"(coefficient of variation: {np.std(fold_sizes)/np.mean(fold_sizes):.2%})")
print()

for feat_name, X in [("full_{}"    .format(len(dp_cols)), X_full), ("filtered_{}".format(len(filtered_cols)), X_filt)]:
    yt, yp, fscores = run_cv(X, species, sgkf, groups=groups, clf=lr)
    bac = balanced_accuracy_score(yt, yp)
    ci  = bootstrap_ci(yt, yp, balanced_accuracy_score)
    f1  = f1_score(yt, yp, average="macro")
    results_q1[(feat_name, "grouped_CV")] = (bac, ci[0], ci[1], fscores)

    label = "★ PRIMARY" if feat_name == "filtered_{}".format(len(filtered_cols)) else ""
    print(f"Feature set: {feat_name}  {label}")
    print(f"  Fold scores: {[round(s, 3) for s in fscores]}")
    print(f"  Mean balanced acc: {bac:.3f}  95% CI: [{ci[0]:.3f}, {ci[1]:.3f}]")
    print(f"  Macro-F1:          {f1:.3f}")
    print()

# Null baseline under grouped CV
yt_n, yp_n, _ = run_cv(X_full, species, sgkf, groups=groups, clf=null_clf)
null_bac = balanced_accuracy_score(yt_n, yp_n)
null_ci  = bootstrap_ci(yt_n, yp_n, balanced_accuracy_score)
print(f"Null baseline (stratified random, grouped CV):")
print(f"  {null_bac:.3f}  95% CI: [{null_ci[0]:.3f}, {null_ci[1]:.3f}]")
print(f"  (Theoretical for perfectly balanced 6-class: {1/6:.3f})")


=== Q1: StratifiedGroupKFold (phylogenetic correction) ===

--- Fold structure ---
  Fold 1:  232 genomes |  24 phylogroups | AB:18  EC:30  EF:119  KP:25  PA:28  SA:12
  Fold 2:  212 genomes |  24 phylogroups | AB:18  EC:29  EF:6  KP:27  PA:28  SA:104
  Fold 3:  178 genomes |  21 phylogroups | AB:76  EC:29  EF:7  KP:26  PA:29  SA:11
  Fold 4:  133 genomes |  12 phylogroups | AB:20  EC:29  EF:11  KP:26  PA:36  SA:11
  Fold 5:  123 genomes |  14 phylogroups | AB:18  EC:29  EF:7  KP:28  PA:29  SA:12

Fold sizes: [232, 212, 178, 133, 123]
  Range: 123–232 genomes  (coefficient of variation: 24.29%)



Feature set: full_274  
  Fold scores: [np.float64(0.997), np.float64(0.973), np.float64(0.944), np.float64(0.964), np.float64(0.985)]
  Mean balanced acc: 0.979  95% CI: [0.970, 0.988]
  Macro-F1:          0.979



Feature set: filtered_266  ★ PRIMARY
  Fold scores: [np.float64(0.907), np.float64(0.888), np.float64(0.824), np.float64(0.796), np.float64(0.919)]
  Mean balanced acc: 0.837  95% CI: [0.813, 0.859]
  Macro-F1:          0.835



Null baseline (stratified random, grouped CV):
  0.130  95% CI: [0.108, 0.153]
  (Theoretical for perfectly balanced 6-class: 0.167)


## Section 8 — Q1: 2×2 summary table

Pre-specified format from `docs/pre_analysis_plan.md §4`.

The **delta** rows show the accuracy drop from standard → grouped CV. This directly quantifies
how much of the preliminary result was due to clone contamination:
- Small delta (< 0.03): standard CV was largely valid; grouped CV confirms it.
- Large delta (> 0.10): substantial clone leakage was inflating the preliminary result.


In [7]:
def fmt_cell(key, tag=""):
    bac, lo, hi, _ = results_q1[key]
    return f"{bac:.3f} [{lo:.3f}–{hi:.3f}]{tag}"

table = pd.DataFrame({
    "Full ({} dp_*)".format(len(dp_cols)): [
        fmt_cell(("full_{}"    .format(len(dp_cols)),    "standard_CV"), "  [ref]"),
        fmt_cell(("full_{}"    .format(len(dp_cols)),    "grouped_CV")),
    ],
    "Filtered (<0.70, {} dp_*)".format(len(filtered_cols)): [
        fmt_cell(("filtered_{}".format(len(filtered_cols)), "standard_CV"), "  [ref]"),
        fmt_cell(("filtered_{}".format(len(filtered_cols)), "grouped_CV"),   "  ★ PRIMARY"),
    ],
}, index=["Standard StratifiedKFold", "Phylogenetic GroupedStratifiedKFold"])

print("Q1 — Balanced Accuracy (mean [95% CI])")
print("=" * 80)
print(table.to_string())
print()

for feat, label in [("full_{}"    .format(len(dp_cols)), "Full"), ("filtered_{}".format(len(filtered_cols)), "Filtered")]:
    std_bac = results_q1[(feat, "standard_CV")][0]
    grp_bac = results_q1[(feat, "grouped_CV")][0]
    delta   = grp_bac - std_bac
    print(f"  {label:10s}: delta (grouped − standard) = {delta:+.3f}  "
          f"({'small: std CV largely valid' if abs(delta) < 0.03 else 'moderate' if abs(delta) < 0.10 else 'large: substantial clone leakage in std CV'})")


Q1 — Balanced Accuracy (mean [95% CI])
                                                Full (274 dp_*)      Filtered (<0.70, 266 dp_*)
Standard StratifiedKFold             0.988 [0.980–0.994]  [ref]      0.950 [0.935–0.964]  [ref]
Phylogenetic GroupedStratifiedKFold         0.979 [0.970–0.988]  0.837 [0.813–0.859]  ★ PRIMARY

  Full      : delta (grouped − standard) = -0.009  (small: std CV largely valid)
  Filtered  : delta (grouped − standard) = -0.114  (large: substantial clone leakage in std CV)


## Section 9 — Q1: Confusion matrix (primary result)

Row-normalised confusion matrix for **[filtered features + grouped CV]** — the primary result.

**How to read this:**
- Rows = true species. Columns = predicted species.
- Each cell = fraction of that species' genomes predicted as that class.
- Diagonal = recall per species (fraction correctly classified).
- Off-diagonal = what each species gets confused with.

A high off-diagonal value between AB and EC (for example) would mean the model struggles to
distinguish those two species based on defence system profile alone — a biologically meaningful
finding about their defence system similarity.


In [8]:
# Collect predictions for primary result (filtered + grouped CV)
yt_prim, yp_prim, fscores_prim = run_cv(X_filt, species, sgkf, groups=groups, clf=lr)

sp_order_plot = sorted(set(species))
cm     = confusion_matrix(yt_prim, yp_prim, labels=sp_order_plot)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_norm, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=[s[:2].upper() for s in sp_order_plot],
    yticklabels=[s[:2].upper() for s in sp_order_plot],
    ax=ax, vmin=0, vmax=1, linewidths=0.5, annot_kws={"size": 11},
)
ax.set_xlabel("Predicted species", fontsize=11)
ax.set_ylabel("True species", fontsize=11)
bac_prim = results_q1[("filtered_{}".format(len(filtered_cols)), "grouped_CV")][0]
ci_prim  = results_q1[("filtered_{}".format(len(filtered_cols)), "grouped_CV")][1:3]
ax.set_title(
    f"Q1 Confusion Matrix — PRIMARY RESULT\n"
    f"Specificity-filtered + Phylogenetic GroupedStratifiedKFold\n"
    f"Balanced acc = {bac_prim:.3f}  [{ci_prim[0]:.3f}–{ci_prim[1]:.3f}]  (row-normalised = recall)",
    fontsize=10,
)
plt.tight_layout()
plt.savefig(FIG / "q1_confusion_matrix_primary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/figures/q1_confusion_matrix_primary.png")

print("\nPer-class recall:")
for i, sp in enumerate(sp_order_plot):
    print(f"  {sp:<15}: {cm_norm[i,i]:.3f}")


Saved: results/figures/q1_confusion_matrix_primary.png

Per-class recall:
  abaumannii     : 0.567
  ecloaceae      : 0.863
  efaecium       : 0.893
  kpneumoniae    : 0.864
  paeruginosa    : 0.893
  saureus        : 0.940


## Section 10 — Q2: ARG burden prediction (per species, GroupedStratifiedKFold)

**The question:** Within each species, can defence system profile predict whether a genome
is in the high-ARG burden group vs the low-ARG burden group?

**Setup:**
- Labels: `low_ARG` = 0, `high_ARG` = 1. `mid_ARG` genomes are excluded (same as the
  middle tertile exclusion in the primary Q2 design).
- Feature set: specificity-filtered (same as the primary Q1 result — consistent across questions).
- CV: StratifiedGroupKFold using only that species' phylogroups. Each species runs entirely
  independently.
- Metrics: balanced accuracy (primary) + AUROC (secondary).

**Why per species?**
ARG burden labels are defined *within each species* (tertile boundaries computed per species).
A cross-species Q2 model would conflate species-level ARG baseline differences with the
within-species defence architecture signal we are trying to measure.

**Why balanced accuracy for Q2?**
Within each species, low_ARG and high_ARG group sizes may not be equal (mid_ARG genomes
excluded; PA uses median split). Balanced accuracy averages recall across both classes,
preventing the model from "winning" by always predicting the larger class.


In [9]:
print("=== Q2: ARG Burden Prediction (StratifiedGroupKFold, per species) ===\n")
print(f"{'Species':<15}  {'N':>5}  {'PGs':>4}  {'BAcc':>6}  {'95% CI':>16}  "
      f"{'AUROC':>6}  {'Null':>6}  {'Beat null?':>11}")
print("-" * 82)

results_q2 = {}

for sp in sorted(fm["species"].unique()):
    # Subset: this species only, low/high ARG only
    mask  = (fm["species"] == sp) & (fm["arg_burden_tertile"].isin(["low_ARG", "high_ARG"]))
    sp_fm = fm[mask]
    sp_X  = sp_fm[filtered_cols].values.astype(np.float32)
    sp_y  = (sp_fm["arg_burden_tertile"] == "high_ARG").astype(int).values
    sp_g  = cv_groups.loc[sp_fm.index, "phylogroup"].values

    n_pg   = len(set(sp_g))
    n_elig = len(sp_fm)

    if n_pg < N_SPLITS:
        print(f"{sp:<15}  {n_elig:>5}  {n_pg:>4}  SKIP — only {n_pg} phylogroups")
        results_q2[sp] = None
        continue

    sgkf_q2  = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    lr_bin   = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs",
                                   max_iter=2000, random_state=RANDOM_STATE)
    null_bin = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)

    # Collect y_true, y_pred, y_proba in one loop
    all_true, all_pred, all_proba = [], [], []
    for tr, te in sgkf_q2.split(sp_X, sp_y, groups=sp_g):
        m = clone(lr_bin)
        m.fit(sp_X[tr], sp_y[tr])
        all_true.extend(sp_y[te])
        all_pred.extend(m.predict(sp_X[te]))
        all_proba.extend(m.predict_proba(sp_X[te])[:, 1])

    yt, yp, yproba = np.array(all_true), np.array(all_pred), np.array(all_proba)
    bac   = balanced_accuracy_score(yt, yp)
    ci    = bootstrap_ci(yt, yp, balanced_accuracy_score)
    auroc = roc_auc_score(yt, yproba)

    # Null
    null_all_true, null_all_pred = [], []
    for tr, te in sgkf_q2.split(sp_X, sp_y, groups=sp_g):
        m = clone(null_bin)
        m.fit(sp_X[tr], sp_y[tr])
        null_all_true.extend(sp_y[te])
        null_all_pred.extend(m.predict(sp_X[te]))
    null_bac = balanced_accuracy_score(np.array(null_all_true), np.array(null_all_pred))

    beat = "YES" if ci[0] > null_bac else ("NO" if ci[1] < null_bac else "borderline")
    results_q2[sp] = dict(n_elig=n_elig, n_pg=n_pg, bac=bac,
                           ci_lo=ci[0], ci_hi=ci[1], auroc=auroc,
                           null_bac=null_bac, yt=yt, yp=yp, yproba=yproba)

    print(f"{sp:<15}  {n_elig:>5}  {n_pg:>4}  {bac:.3f}  [{ci[0]:.3f}–{ci[1]:.3f}]  "
          f"{auroc:.3f}  {null_bac:.3f}  {beat:>11}")

print(f"\nTheoretical null for balanced binary: 0.500")
print(f"Beat null criterion: lower 95% CI bound > null balanced accuracy")


=== Q2: ARG Burden Prediction (StratifiedGroupKFold, per species) ===

Species              N   PGs    BAcc            95% CI   AUROC    Null   Beat null?
----------------------------------------------------------------------------------


abaumannii         101    13  0.473  [0.422–0.522]  0.231  0.358          YES


ecloaceae           97    20  0.752  [0.659–0.837]  0.846  0.504          YES


efaecium           104     7  0.512  [0.461–0.567]  0.578  0.397          YES


kpneumoniae         86    16  0.719  [0.618–0.811]  0.830  0.465          YES


paeruginosa        120    26  0.645  [0.558–0.732]  0.698  0.435          YES


saureus            106     9  0.470  [0.406–0.533]  0.556  0.498   borderline

Theoretical null for balanced binary: 0.500
Beat null criterion: lower 95% CI bound > null balanced accuracy


## Section 11 — Save results

In [10]:
# Q1 results
q1_rows = []
for (feat, cv_nm), (bac, lo, hi, fsc) in results_q1.items():
    q1_rows.append(dict(
        feature_set=feat, cv_strategy=cv_nm,
        balanced_acc=round(bac, 4),
        ci_lo=round(lo, 4), ci_hi=round(hi, 4),
        fold_scores=[round(float(s), 4) for s in fsc],
    ))
pd.DataFrame(q1_rows).to_parquet(ROOT / "results" / "q1_lr_results_3460.parquet")
print("Saved: results/q1_lr_results_3460.parquet")

# Q2 results
q2_rows = []
for sp, r in results_q2.items():
    if r is None:
        continue
    q2_rows.append(dict(
        species=sp,
        n_eligible=r["n_elig"], n_phylogroups=r["n_pg"],
        balanced_acc=round(r["bac"], 4),
        ci_lo=round(r["ci_lo"], 4), ci_hi=round(r["ci_hi"], 4),
        auroc=round(r["auroc"], 4),
        null_bac=round(r["null_bac"], 4),
    ))
pd.DataFrame(q2_rows).sort_values("balanced_acc", ascending=False).to_parquet(
    ROOT / "results" / "q2_lr_results.parquet"
)
print("Saved: results/q2_lr_results.parquet")
print("\n=== Phase 7 complete ===")
print("Next: Phase 8 — Random Forest (notebooks/06_random_forest.ipynb)")


Saved: results/q1_lr_results.parquet
Saved: results/q2_lr_results.parquet

=== Phase 7 complete ===
Next: Phase 8 — Random Forest (notebooks/06_random_forest.ipynb)
